# 03 - The one train/test split

**Input:** `data/hpv_clean.csv`. **Output:** `data/split.csv`.

We make the split once, here. Every modelling notebook loads `data/split.csv`. Nobody makes another split, because scores from different splits cannot be compared.

**Our decision (27 November):** 80% train, 20% test, stratified by class **and** by division. The test set then has the same class shares and the same share of each division as the whole data.

The second test, leave one division out, needs no file. It uses the column `meta_division`.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
TEST_SIZE = 0.20

df = pd.read_csv("../data/hpv_clean.csv")
df.shape

(2147, 101)

## The strata

A stratum is one combination of division and class, for example "Dhaka, Only if free".

Some combinations are very small. Mymensingh has one parent in "Only if free". A group of one parent cannot be shared between train and test. So every combination with fewer than 10 parents is put in one stratum for its class.

In [2]:
key = df["meta_division"] + " | " + df["target_3class"]
size = key.map(key.value_counts())
strata = key.where(size >= 10, "small divisions | " + df["target_3class"])

print("Number of strata:", strata.nunique())
strata.value_counts().tail(6)

Number of strata: 20


Barishal | Undecided or no           20
Barishal | Only if free              19
Rajshahi | Undecided or no           14
Khulna | Undecided or no             12
small divisions | Undecided or no    11
small divisions | Only if free       10
Name: count, dtype: int64

## The split

In [3]:
train_rows, test_rows = train_test_split(df.index, test_size=TEST_SIZE, stratify=strata, random_state=SEED)

split = pd.DataFrame({"row": df.index, "set": "train"})
split.loc[test_rows, "set"] = "test"
split["set"].value_counts()

set
train    1717
test      430
Name: count, dtype: int64

## Check: the train set and the test set have the same shares

In [4]:
CLASSES = ["Will vaccinate", "Only if free", "Undecided or no"]
df["set"] = split["set"]

by_class = (pd.crosstab(df["set"], df["target_3class"], normalize="index") * 100).round(1)[CLASSES]
by_division = (pd.crosstab(df["set"], df["meta_division"], normalize="index") * 100).round(1)

display(by_class)
display(by_division)

assert (by_class.loc["train"] - by_class.loc["test"]).abs().max() < 1.0
assert (by_division.loc["train"] - by_division.loc["test"]).abs().max() < 1.0
assert set(split["set"]) == {"train", "test"} and len(split) == len(df) == 2147

target_3class,Will vaccinate,Only if free,Undecided or no
set,,,
test,70.5,16.5,13.0
train,70.2,16.5,13.3


meta_division,Barishal,Chattogram,Dhaka,Khulna,Mymensingh,Rajshahi,Rangpur,Sylhet
set,,,,,,,,
test,6.0,20.2,26.0,10.7,7.4,12.3,10.7,6.5
train,5.9,20.2,26.1,10.8,7.5,12.3,10.6,6.7


## Save

In [5]:
split.to_csv("../data/split.csv", index=False)
check = pd.read_csv("../data/split.csv")
print("Saved ../data/split.csv |", (check["set"] == "train").sum(), "train rows,", (check["set"] == "test").sum(), "test rows")
check.head()

Saved ../data/split.csv | 1717 train rows, 430 test rows


,row,set
0,0,test
1,1,train
2,2,train
3,3,test
4,4,train


## How every other notebook uses it

`row` is the position of the parent in `data/hpv_clean.csv`, counted from 0.

```python
df = pd.read_csv("../data/hpv_clean.csv")
split = pd.read_csv("../data/split.csv")
train = df[split["set"] == "train"]
test = df[split["set"] == "test"]
```